Proses Agregasi Data — Forecasting
1. **Forecasting Revenue** — prediksi penjualan distributor per minggu
2. **Optimalisasi Stok** — fitur untuk model demand-planning per gudang & material

#### 1. Setup & Load Data

In [1]:
import pandas as pd
import numpy as np
from itertools import product

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [2]:
df_sell_in  = pd.read_csv('table_sell_in_new.csv',  parse_dates=['date'])
df_sell_out = pd.read_csv('table_sell_out_new.csv', parse_dates=['tanggal_trx'])
df_ca       = pd.read_csv('table_CA_new.csv',       parse_dates=['tanggal_trx'])
df_survey   = pd.read_csv('table_survey_new.csv',   parse_dates=['visit_date'])

print(f"sell_in  : {df_sell_in.shape}")
print(f"sell_out : {df_sell_out.shape}")
print(f"ca       : {df_ca.shape}")
print(f"survey   : {df_survey.shape}")

sell_in  : (15439, 24)
sell_out : (49167, 9)
ca       : (15383, 30)
survey   : (8350, 36)


##### 1.1 · Verifikasi satuan `quantity` pada sell-in

`baseunit_of_measure` pada sell-in berisi label kemasan (`BAG` = 40 kg, `ZAK` = 50 kg).
Namun nilai `quantity` perlu diverifikasi apakah sudah dikonversi ke **ton** atau masih
dalam satuan sak/bag.

In [3]:
avg_price_per_qty = df_sell_in['revenue'].sum() / df_sell_in['quantity'].sum()
print(f"Rata-rata harga per unit quantity : Rp {avg_price_per_qty:,.0f}")

if 800_000 <= avg_price_per_qty <= 1_600_000:
    print("✓  quantity sudah dalam TON — tidak perlu konversi.")
    print("   (Label BAG/ZAK hanya menunjukkan jenis kemasan, bukan satuan quantity)")
else:
    print("⚠  quantity BUKAN dalam ton. Perlu konversi:")
    conv = {'BAG': 0.040, 'ZAK': 0.050}   # sesuaikan jika perlu
    print(f"   Faktor konversi yang akan dipakai: {conv}")

Rata-rata harga per unit quantity : Rp 1,126,113
✓  quantity sudah dalam TON — tidak perlu konversi.
   (Label BAG/ZAK hanya menunjukkan jenis kemasan, bukan satuan quantity)


#### 2 · Forecasting Revenue

**Granularitas**: Distributor (`soldto`) × Minggu
**Target (Y)**: `target_revenue` = total revenue sell-in per distributor per minggu
**Anchor table**: `sell_in` — semua tabel digabungkan ke sini dengan LEFT JOIN

##### 2.1 · Pemetaan periode mingguan

In [4]:
def to_week_start(s: pd.Series) -> pd.Series:
    return s.dt.to_period('W').apply(lambda r: r.start_time).dt.normalize()

df_sell_in['periode']  = to_week_start(df_sell_in['date'])
df_sell_out['periode'] = to_week_start(df_sell_out['tanggal_trx'])
df_ca['periode']       = to_week_start(df_ca['tanggal_trx'])
df_survey['periode']   = to_week_start(df_survey['visit_date'])

print(f"Rentang periode: {df_sell_in['periode'].min().date()} — {df_sell_in['periode'].max().date()}")
print(f"Jumlah minggu  : {df_sell_in['periode'].nunique()}")

Rentang periode: 2025-12-29 — 2026-12-28
Jumlah minggu  : 53


##### 2.2 · Agregasi tabel sumber

In [5]:
# ── Sell-in: anchor tabel (level: soldto × periode) ──────────────────────────
sell_in_agg = df_sell_in.groupby(['soldto', 'periode']).agg(
    target_revenue     = ('revenue',  'sum'),
    total_qty_sellin   = ('quantity', 'sum'),
).reset_index()

# ── Sell-out: daya serap pasar (level: kode_distributor × periode) ────────────
sell_out_agg = df_sell_out.groupby(['kode_distributor', 'periode']).agg(
    total_tonase_sellout = ('sum(tonase)', 'sum'),
    jumlah_toko_aktif    = ('kode_toko',   'nunique'),
).reset_index()

# ── CA: kapasitas gudang distributor (level: kode_distributor × periode) ──────
ca_agg = df_ca.groupby(['kode_distributor', 'periode']).agg(
    total_kapasitas_tonase     = ('tonase_total', 'sum'),
    jumlah_gudang_distributor  = ('kode_gudang',  'nunique'),
).reset_index()

print("sell_in_agg  :", sell_in_agg.shape)
print("sell_out_agg :", sell_out_agg.shape)
print("ca_agg       :", ca_agg.shape)

sell_in_agg  : (5057, 4)
sell_out_agg : (5057, 4)
ca_agg       : (5057, 4)


##### 2.3 · Agregasi survey ke level distributor *(fitur baru)*

Survey mencatat kondisi **lapangan di toko-toko** yang dilayani oleh setiap distributor.
Fitur utama yang relevan:
- `avg_stock_toko` — stok rata-rata di toko; jika rendah → sinyal distributor akan segera restock
- `total_volume_sales` — volume penjualan aktual toko (real demand)
- `total_volume_order` — total order yang diminta toko (leading indicator permintaan mendatang)
- `n_toko_surveyed` — cakupan kunjungan salesman

In [6]:
survey_agg = df_survey.groupby(['distributor_code', 'periode']).agg(
    avg_stock_toko       = ('stock',          'mean'),
    total_volume_sales   = ('volume_sales',   'sum'),
    total_volume_order   = ('volume_order',   'sum'),
    avg_kapasitas_gudang = ('kapasitas_gudang','mean'),
    n_toko_surveyed      = ('customer_code',  'nunique'),
).reset_index()
survey_agg.rename(columns={'distributor_code': 'soldto'}, inplace=True)

print("survey_agg :", survey_agg.shape)
survey_agg.head(3)

survey_agg : (4195, 7)


,soldto,periode,avg_stock_toko,total_volume_sales,total_volume_order,avg_kapasitas_gudang,n_toko_surveyed
0,SD-1010-01,2025-12-29,224.50,101,277,961.00,2
1,SD-1010-01,2026-01-05,130.00,219,202,961.00,3
2,SD-1010-01,2026-01-12,188.50,223,372,"1,469.00",3


##### 2.4 · Pipeline merge

In [7]:
# Step 1: sell_in ← sell_out
df_rev = pd.merge(
    sell_in_agg,
    sell_out_agg,
    left_on=['soldto', 'periode'],
    right_on=['kode_distributor', 'periode'],
    how='left'
).drop(columns='kode_distributor')

df_rev[['total_tonase_sellout', 'jumlah_toko_aktif']] = \
    df_rev[['total_tonase_sellout', 'jumlah_toko_aktif']].fillna(0)

# Step 2: ← CA
df_rev = pd.merge(
    df_rev,
    ca_agg,
    left_on=['soldto', 'periode'],
    right_on=['kode_distributor', 'periode'],
    how='left'
).drop(columns='kode_distributor', errors='ignore')

df_rev[['total_kapasitas_tonase', 'jumlah_gudang_distributor']] = \
    df_rev[['total_kapasitas_tonase', 'jumlah_gudang_distributor']].fillna(0)

# Step 3: ← Survey
df_rev = pd.merge(df_rev, survey_agg, on=['soldto', 'periode'], how='left')

# Step 4: ← Info regional (master data dari sell_in)
region_map = df_sell_in[['soldto', 'regional', 'province_desc']].drop_duplicates(subset=['soldto'])
df_rev = pd.merge(df_rev, region_map, on='soldto', how='left')

print(f"Shape setelah merge: {df_rev.shape}")

Shape setelah merge: (5057, 15)


##### 2.5 · Rekayasa fitur

A · Fitur harga & kapasitas

In [8]:
df_rev['avg_price_per_ton'] = df_rev['target_revenue'] / (df_rev['total_qty_sellin'] + 1e-5)
df_rev['utilisasi_kapasitas'] = df_rev['total_qty_sellin'] / (df_rev['total_kapasitas_tonase'] + 1e-5)

B · Lag & rolling fitur `target_revenue`
Nilai historis target sendiri adalah prediktor utama.
Semua fitur di bawah menggunakan `.shift()` agar **tidak ada kebocoran data**.

In [9]:
df_rev = df_rev.sort_values(['soldto', 'periode']).reset_index(drop=True)

# Lag nilai revenue itu sendiri
df_rev['lag_1w_revenue'] = df_rev.groupby('soldto')['target_revenue'].shift(1)
df_rev['lag_4w_revenue'] = df_rev.groupby('soldto')['target_revenue'].shift(4)

# Rolling mean & std DISHIFT dulu → tidak ada kebocoran nilai periode t
df_rev['rolling_4w_revenue_mean'] = (
    df_rev.groupby('soldto')['target_revenue']
    .transform(lambda x: x.shift(1).rolling(4, min_periods=2).mean())
)
df_rev['rolling_4w_revenue_std'] = (
    df_rev.groupby('soldto')['target_revenue']
    .transform(lambda x: x.shift(1).rolling(4, min_periods=2).std())
)
df_rev['rolling_4w_revenue_std'] = df_rev['rolling_4w_revenue_std'].fillna(0)

# Lag sell_out & toko aktif (sinyal daya serap periode sebelumnya)
df_rev['lag_1w_sellout']     = df_rev.groupby('soldto')['total_tonase_sellout'].shift(1)
df_rev['lag_1w_toko_aktif']  = df_rev.groupby('soldto')['jumlah_toko_aktif'].shift(1)

C · Sell-through rate

STR mengukur seberapa cepat stok distributor terserap ke toko.
Secara kausal: STR tinggi di minggu **t** → distributor akan restock lebih banyak di minggu **t+1**.
Maka yang tepat digunakan sebagai prediktor adalah **nilai STR periode sebelumnya**.

In [10]:
# Hitung STR periode saat ini (boleh disimpan sebagai referensi)
df_rev['sell_through_rate_raw'] = (
    df_rev['total_tonase_sellout'] / (df_rev['total_qty_sellin'] + 1e-5)
)

# Lag STR 1 minggu → digunakan sebagai prediktor (tidak ada kebocoran)
df_rev['lag_1w_sell_through_rate'] = (
    df_rev.groupby('soldto')['sell_through_rate_raw'].shift(1)
)

# Rolling STR 4 minggu (juga dishift)
df_rev['rolling_4w_str_mean'] = (
    df_rev.groupby('soldto')['sell_through_rate_raw']
    .transform(lambda x: x.shift(1).rolling(4, min_periods=2).mean())
)

D · Fitur survey

In [11]:
# Survey dilag 1 minggu — kondisi toko minggu lalu memprediksi kebutuhan restock minggu ini
df_rev['lag_1w_avg_stock_toko']  = df_rev.groupby('soldto')['avg_stock_toko'].shift(1)
df_rev['lag_1w_volume_sales']    = df_rev.groupby('soldto')['total_volume_sales'].shift(1)
df_rev['lag_1w_volume_order']    = df_rev.groupby('soldto')['total_volume_order'].shift(1)
df_rev['lag_1w_n_toko_surveyed'] = df_rev.groupby('soldto')['n_toko_surveyed'].shift(1)

# Isi NaN survey dengan 0 (minggu tanpa kunjungan = tidak ada data lapangan)
survey_cols = [c for c in df_rev.columns if c.startswith('lag_1w_') and
               c in ['lag_1w_avg_stock_toko','lag_1w_volume_sales',
                     'lag_1w_volume_order','lag_1w_n_toko_surveyed']]
df_rev[survey_cols] = df_rev[survey_cols].fillna(0)

E · Fitur waktu (musiman)

In [12]:
df_rev['month']     = df_rev['periode'].dt.month
df_rev['quarter']   = df_rev['periode'].dt.quarter
df_rev['week_of_year'] = df_rev['periode'].dt.isocalendar().week.astype(int)

# Cyclical encoding: menghindari diskontinuitas antara bulan 12 dan 1
df_rev['month_sin'] = np.sin(2 * np.pi * df_rev['month'] / 12)
df_rev['month_cos'] = np.cos(2 * np.pi * df_rev['month'] / 12)
df_rev['week_sin']  = np.sin(2 * np.pi * df_rev['week_of_year'] / 52)
df_rev['week_cos']  = np.cos(2 * np.pi * df_rev['week_of_year'] / 52)

##### 2.6 · Finalisasi & ringkasan kolom

In [13]:
# Kolom final yang relevan untuk forecasting
COLS_IDENTIFIER = ['soldto', 'periode', 'regional', 'province_desc']
COLS_TARGET = ['target_revenue']
COLS_SELLIN = ['total_qty_sellin', 'avg_price_per_ton', 'utilisasi_kapasitas']

COLS_LAG_REVENUE = [
    'lag_1w_revenue', 'lag_4w_revenue',
    'rolling_4w_revenue_mean', 'rolling_4w_revenue_std'
]
COLS_SELLOUT = [
    'total_kapasitas_tonase', 'jumlah_gudang_distributor',
    'lag_1w_sellout', 'lag_1w_toko_aktif',
    'lag_1w_sell_through_rate', 'rolling_4w_str_mean'
]
COLS_SURVEY = [
    'lag_1w_avg_stock_toko', 'lag_1w_volume_sales',
    'lag_1w_volume_order', 'lag_1w_n_toko_surveyed'
]
COLS_TEMPORAL = ['month', 'quarter', 'week_of_year',
                 'month_sin', 'month_cos', 'week_sin', 'week_cos']

ALL_COLS = (COLS_IDENTIFIER + COLS_TARGET + COLS_SELLIN +
            COLS_LAG_REVENUE + COLS_SELLOUT + COLS_SURVEY + COLS_TEMPORAL)

df_rev_final = df_rev[ALL_COLS].copy()

print(f"Shape dataset revenue forecasting: {df_rev_final.shape}")
print(f"\nNaN per kolom kritis (lag fitur — wajar di minggu-minggu awal):")
lag_cols = COLS_LAG_REVENUE + ['lag_1w_sell_through_rate']
print(df_rev_final[lag_cols].isna().sum())

Shape dataset revenue forecasting: (5057, 29)

NaN per kolom kritis (lag fitur — wajar di minggu-minggu awal):
lag_1w_revenue              102
lag_4w_revenue              408
rolling_4w_revenue_mean     204
rolling_4w_revenue_std        0
lag_1w_sell_through_rate    102
dtype: int64


In [14]:
df_rev_final.head(5)

,soldto,periode,regional,province_desc,target_revenue,total_qty_sellin,avg_price_per_ton,utilisasi_kapasitas,lag_1w_revenue,lag_4w_revenue,rolling_4w_revenue_mean,rolling_4w_revenue_std,total_kapasitas_tonase,jumlah_gudang_distributor,lag_1w_sellout,lag_1w_toko_aktif,lag_1w_sell_through_rate,rolling_4w_str_mean,lag_1w_avg_stock_toko,lag_1w_volume_sales,lag_1w_volume_order,lag_1w_n_toko_surveyed,month,quarter,week_of_year,month_sin,month_cos,week_sin,week_cos
0,SD-1010-01,2025-12-29,AREA 1,ACEH,41412683,32.18,"1,286,907.09",1.15,NaN,NaN,NaN,0.00,28.07,1,NaN,NaN,NaN,NaN,0.00,0.00,0.00,0.00,12,4,1,-0.00,1.00,0.12,0.99
1,SD-1010-01,2026-01-05,AREA 1,ACEH,107266897,100.83,"1,063,839.00",1.15,"41,412,683.00",NaN,NaN,0.00,87.58,1,29.42,3.00,0.91,NaN,224.50,101.00,277.00,2.00,1,1,2,0.50,0.87,0.24,0.97
2,SD-1010-01,2026-01-12,AREA 1,ACEH,445748792,404.03,"1,103,256.64",1.16,"107,266,897.00",NaN,"74,339,790.00","46,565,961.29",348.94,2,93.17,8.00,0.92,0.92,130.00,219.00,202.00,3.00,1,1,3,0.50,0.87,0.35,0.94
3,SD-1010-01,2026-01-19,AREA 1,ACEH,285072002,246.69,"1,155,587.95",1.05,"445,748,792.00",NaN,"198,142,790.67","216,946,406.51",235.16,2,361.22,16.00,0.89,0.91,188.50,223.00,372.00,3.00,1,1,4,0.50,0.87,0.46,0.89
4,SD-1010-01,2026-01-26,AREA 1,ACEH,349358535,329.36,"1,060,719.35",1.02,"285,072,002.00","41,412,683.00","219,875,093.50","182,390,608.71",323.81,2,228.51,13.00,0.93,0.91,0.00,0.00,0.00,0.00,1,1,5,0.50,0.87,0.57,0.82


In [15]:
df_rev_final.to_csv('data_for_forecast_revenue.csv', index=False)
print("Tersimpan: data_for_forecast_revenue.csv")

Tersimpan: data_for_forecast_revenue.csv


#### 3. Optimalisasi Stok

**Granularitas**: Gudang (`kode_gudang`) × Material (`material_code`) × Minggu
**Target (Y)**: `actual_tonase_in` — volume riil yang masuk ke gudang per minggu per material
**Tujuan**: Fitur untuk model prediksi demand per gudang-material, dilengkapi komponen safety stock

##### 3.1 · Pemetaan kode material & periode mingguan

In [16]:
def map_material_code(product_name: str) -> str:
    """Mapping nama produk CA ke kode material baku (konsisten dengan sell_in)."""
    name = str(product_name).lower()
    if 'tipe i' in name or 'standar' in name:
        return 'MAT001'
    elif 'pcc' in name or 'premium' in name:
        return 'MAT002'
    elif 'khusus' in name or 'tecton' in name:
        return 'MAT003'
    elif 'komposit' in name:
        return 'MAT004'
    return 'MAT_OTHER'

df_ca['material_code'] = df_ca['nama_produk'].apply(map_material_code)

# Periode mingguan
df_sell_in['week_start'] = to_week_start(df_sell_in['date'])
df_ca['week_start']      = to_week_start(df_ca['tanggal_trx'])
df_sell_out['week_start'] = to_week_start(df_sell_out['tanggal_trx'])
df_survey['week_start']  = to_week_start(df_survey['visit_date'])

##### 3.2 · Pemetaan gudang ke provinsi & skeleton grid

In [17]:
# Master peta gudang → provinsi (dari sell_in sebagai referensi)
node_map = (df_sell_in[['shipto', 'province_desc']]
            .drop_duplicates()
            .rename(columns={'shipto': 'kode_gudang', 'province_desc': 'provinsi'}))

unique_weeks     = pd.date_range(
    start=df_sell_in['week_start'].min(),
    end=df_sell_in['week_start'].max(),
    freq='W-MON'
)
unique_gudang    = node_map['kode_gudang'].unique()
unique_materials = ['MAT001', 'MAT002', 'MAT003', 'MAT004']

# Cartesian product: semua kombinasi (minggu × gudang × material)
master_ts = pd.DataFrame(
    list(product(unique_weeks, unique_gudang, unique_materials)),
    columns=['week_start', 'kode_gudang', 'material_code']
)
master_ts = pd.merge(master_ts, node_map, on='kode_gudang', how='left')

print(f"Skeleton grid: {master_ts.shape}  "
      f"({len(unique_weeks)}w × {len(unique_gudang)}gudang × {len(unique_materials)}mat)")

Skeleton grid: (14416, 4)  (53w × 68gudang × 4mat)


##### 3.3 · Agregasi data fakta

In [18]:
# ── a. Aktual pasokan gudang (dari sell-in) ───────────────────────────────────
agg_actual = (df_sell_in
              .groupby(['week_start', 'shipto', 'material_code'])['quantity']
              .sum().reset_index()
              .rename(columns={'shipto': 'kode_gudang', 'quantity': 'actual_tonase_in'}))

# ── b. Target alokasi gudang (dari CA) ───────────────────────────────────────
agg_target = (df_ca
              .groupby(['week_start', 'kode_gudang', 'material_code'])['tonase_total']
              .sum().reset_index()
              .rename(columns={'tonase_total': 'target_tonase_ca'}))

print(f"agg_actual : {agg_actual.shape}")
print(f"agg_target : {agg_target.shape}")

agg_actual : (9413, 4)
agg_target : (9379, 4)


##### Sell-out per gudang
Meski tabel sell-out tidak memiliki `material_code`, **total sell-out per gudang per minggu**
tetap merupakan sinyal permintaan yang valid — mencerminkan total throughput gudang tersebut
sebagai ukuran seberapa besar toko-toko di sekitarnya menyerap produk.

In [19]:
agg_sellout_gudang = (df_sell_out
                       .groupby(['week_start', 'kode_gudang'])['sum(tonase)']
                       .sum().reset_index()
                       .rename(columns={'sum(tonase)': 'total_sellout_gudang'}))

print(f"agg_sellout_gudang : {agg_sellout_gudang.shape}")

agg_sellout_gudang : (3550, 3)


##### Survey per gudang

Data survey tersedia di level distributor. Karena setiap distributor memiliki tepat
**2 gudang** (terverifikasi dari data), fitur survey di-broadcast ke kedua gudang
distributor tersebut. Ini valid karena kedua gudang berada dalam lingkup operasi
distributor yang sama dan dilayani salesman yang sama.

In [20]:
# Peta distributor → gudang (1 distributor = 2 gudang, konsisten di seluruh data)
dist_gudang_map = (df_sell_in[['soldto', 'shipto']]
                   .drop_duplicates()
                   .rename(columns={'soldto': 'distributor_code', 'shipto': 'kode_gudang'}))

# Agregasi survey ke level distributor-minggu
survey_dist_agg = (df_survey
                   .groupby(['distributor_code', 'week_start']).agg(
                       avg_stock_toko_dist   = ('stock',          'mean'),
                       total_vol_sales_dist  = ('volume_sales',   'sum'),
                       total_vol_order_dist  = ('volume_order',   'sum'),
                       n_toko_dist           = ('customer_code',  'nunique'),
                   ).reset_index())

# Broadcast ke gudang (distributor × gudang mapping)
survey_gudang_agg = pd.merge(dist_gudang_map, survey_dist_agg,
                              on='distributor_code', how='left')

# Deduplicate survey_gudang_agg to prevent master_ts row duplication
survey_gudang_agg = (survey_gudang_agg
                     .groupby(['week_start', 'kode_gudang']).agg(
                         distributor_code      = ('distributor_code', 'first'),
                         avg_stock_toko_dist   = ('avg_stock_toko_dist', 'mean'),
                         total_vol_sales_dist  = ('total_vol_sales_dist', 'sum'),
                         total_vol_order_dist  = ('total_vol_order_dist', 'sum'),
                         n_toko_dist           = ('n_toko_dist', 'sum'),
                     ).reset_index())

print(f"survey_gudang_agg : {survey_gudang_agg.shape}")

survey_gudang_agg : (3548, 7)


##### 3.4 · Merge ke skeleton grid

In [21]:
master_ts = pd.merge(master_ts, agg_actual,
                     on=['week_start', 'kode_gudang', 'material_code'], how='left')
master_ts = pd.merge(master_ts, agg_target,
                     on=['week_start', 'kode_gudang', 'material_code'], how='left')
master_ts = pd.merge(master_ts, agg_sellout_gudang,
                     on=['week_start', 'kode_gudang'], how='left')
master_ts = pd.merge(master_ts, survey_gudang_agg,
                     on=['week_start', 'kode_gudang'], how='left')

# Isi NaN dengan 0 (tidak ada pasokan/target/data di minggu tersebut)
fill_zero_cols = ['actual_tonase_in', 'target_tonase_ca',
                  'total_sellout_gudang',
                  'avg_stock_toko_dist', 'total_vol_sales_dist',
                  'total_vol_order_dist', 'n_toko_dist']
master_ts[fill_zero_cols] = master_ts[fill_zero_cols].fillna(0)

master_ts = master_ts.sort_values(['kode_gudang', 'material_code', 'week_start']).reset_index(drop=True)
print(f"master_ts shape setelah merge: {master_ts.shape}")

master_ts shape setelah merge: (14416, 12)


#### 3.5 · Rekayasa fitur

##### A · Lag & rolling

In [22]:
grp = master_ts.groupby(['kode_gudang', 'material_code'])

# Lag aktual
master_ts['lag_1w_actual'] = grp['actual_tonase_in'].shift(1)
master_ts['lag_2w_actual'] = grp['actual_tonase_in'].shift(2)
master_ts['lag_4w_actual'] = grp['actual_tonase_in'].shift(4)

# Rolling mean — shift(1) DULU, baru rolling (tidak ada kebocoran)
master_ts['rolling_4w_actual_mean'] = (
    grp['actual_tonase_in']
    .transform(lambda x: x.shift(1).rolling(4, min_periods=2).mean())
)

# Rolling std — sama: shift(1) dulu
master_ts['rolling_4w_actual_std'] = (
    grp['actual_tonase_in']
    .transform(lambda x: x.shift(1).rolling(4, min_periods=2).std())
)
master_ts['rolling_4w_actual_std'] = master_ts['rolling_4w_actual_std'].fillna(0)

# Lag sell-out gudang (demand signal hilir, dilag 1 minggu)
master_ts['lag_1w_sellout_gudang'] = grp['total_sellout_gudang'].shift(1)

# Lag fitur survey (kondisi toko minggu lalu)
master_ts['lag_1w_stock_toko']     = grp['avg_stock_toko_dist'].shift(1)
master_ts['lag_1w_vol_order_dist'] = grp['total_vol_order_dist'].shift(1)

##### B · Fulfillment ratio

Mengukur seberapa besar target CA terpenuhi oleh pasokan aktual. Sebagai prediktor,
nilai periode *t-1* yang digunakan.

In [23]:
lag_actual = grp['actual_tonase_in'].shift(1)
lag_target = grp['target_tonase_ca'].shift(1)

master_ts['lag_1w_fulfillment_ratio'] = lag_actual / (lag_target + 1e-5)

##### C · Komponen Safety Stock

Formula klasik: **SS = Z × σ × √(lead_time)**

- **Z** — Z-score dari service level target (misal 1.645 untuk 95%)
- **σ** — standar deviasi demand (sudah ada: `rolling_4w_actual_std`)
- **lead_time** — minggu rata-rata dari pemesanan ke penerimaan di gudang

> Sesuaikan `LEAD_TIME_WEEKS` dan `SERVICE_LEVEL_Z` dengan parameter bisnis aktual.
> Nilai *safety stock* ini tidak langsung menjadi target, melainkan **fitur** yang
> memberi model konteks tentang kebutuhan buffer minimum setiap gudang-material.

In [24]:
# ── Parameter bisnis — sesuaikan dengan kondisi aktual ───────────────────────
LEAD_TIME_WEEKS = 1       # rata-rata lead time pabrik → gudang (dalam minggu)
SERVICE_LEVEL_Z = 1.645   # Z-score untuk service level 95%

# Komponen safety stock
master_ts['safety_stock_est'] = (
    SERVICE_LEVEL_Z
    * master_ts['rolling_4w_actual_std']
    * np.sqrt(LEAD_TIME_WEEKS)
)

# Reorder Point estimasi: rata-rata demand × lead time + safety stock
master_ts['reorder_point_est'] = (
    master_ts['rolling_4w_actual_mean'] * LEAD_TIME_WEEKS
    + master_ts['safety_stock_est']
)

# Utilisasi vs kapasitas CA (apakah pasokan sudah mencapai target alokasi?)
master_ts['utilisasi_vs_ca'] = (
    master_ts['actual_tonase_in'] / (master_ts['target_tonase_ca'] + 1e-5)
)

##### D · Fitur waktu (musiman)

In [25]:
master_ts['month']        = master_ts['week_start'].dt.month
master_ts['quarter']      = master_ts['week_start'].dt.quarter
master_ts['week_of_year'] = master_ts['week_start'].dt.isocalendar().week.astype(int)
master_ts['month_sin']    = np.sin(2 * np.pi * master_ts['month'] / 12)
master_ts['month_cos']    = np.cos(2 * np.pi * master_ts['month'] / 12)
master_ts['week_sin']     = np.sin(2 * np.pi * master_ts['week_of_year'] / 52)
master_ts['week_cos']     = np.cos(2 * np.pi * master_ts['week_of_year'] / 52)

##### 3.6 · Drop baris NaN awal & quality check

In [26]:
n_before = len(master_ts)

# Kolom kunci yang wajib tersedia (lag minimal 1 minggu)
required_cols = ['lag_1w_actual', 'lag_2w_actual', 'rolling_4w_actual_mean']
master_ts = master_ts.dropna(subset=required_cols).reset_index(drop=True)

n_after = len(master_ts)
print(f"Baris sebelum dropna : {n_before:,}")
print(f"Baris setelah dropna : {n_after:,}  (drop {n_before - n_after:,} baris awal per grup)")
print(f"\nDimensi final: {master_ts.shape}")

# NaN yang tersisa (boleh ada: kolom survey yang minggu tidak ada kunjungan)
remaining_nan = master_ts.isna().sum()
remaining_nan = remaining_nan[remaining_nan > 0]
if len(remaining_nan):
    print(f"\nNaN tersisa (isi 0 jika diperlukan):")
    print(remaining_nan)
else:
    print("\n✓  Tidak ada NaN kritis tersisa.")

Baris sebelum dropna : 14,416
Baris setelah dropna : 13,872  (drop 544 baris awal per grup)

Dimensi final: (13872, 31)

NaN tersisa (isi 0 jika diperlukan):
distributor_code    192
lag_4w_actual       544
dtype: int64


In [27]:
# Isi NaN survey (minggu tanpa kunjungan → 0)
survey_feat_cols = ['lag_1w_sellout_gudang', 'lag_1w_stock_toko', 'lag_1w_vol_order_dist']
master_ts[survey_feat_cols] = master_ts[survey_feat_cols].fillna(0)

print("Shape final:", master_ts.shape)
master_ts.head(5)

Shape final: (13872, 31)


,week_start,kode_gudang,material_code,provinsi,actual_tonase_in,target_tonase_ca,total_sellout_gudang,distributor_code,avg_stock_toko_dist,total_vol_sales_dist,total_vol_order_dist,n_toko_dist,lag_1w_actual,lag_2w_actual,lag_4w_actual,rolling_4w_actual_mean,rolling_4w_actual_std,lag_1w_sellout_gudang,lag_1w_stock_toko,lag_1w_vol_order_dist,lag_1w_fulfillment_ratio,safety_stock_est,reorder_point_est,utilisasi_vs_ca,month,quarter,week_of_year,month_sin,month_cos,week_sin,week_cos
0,2026-01-12,WH-1010-01,MAT001,ACEH,0.00,0.00,477.84,SD-1010-01,99.75,315.00,430.00,4.00,53.28,0.00,NaN,26.64,37.67,140.28,150.17,570.00,1.00,61.97,88.61,0.00,1,1,3,0.50,0.87,0.35,0.94
1,2026-01-19,WH-1010-01,MAT001,ACEH,0.00,0.00,402.78,SD-1010-02,144.50,141.00,214.00,2.00,0.00,53.28,NaN,17.76,30.76,477.84,99.75,430.00,0.00,50.60,68.36,0.00,1,1,4,0.50,0.87,0.46,0.89
2,2026-01-26,WH-1010-01,MAT001,ACEH,210.96,205.11,386.18,SD-1010-02,227.67,225.00,547.00,4.00,0.00,0.00,0.00,13.32,26.64,402.78,144.50,214.00,0.00,43.82,57.14,1.03,1,1,5,0.50,0.87,0.57,0.82
3,2026-02-02,WH-1010-01,MAT001,ACEH,0.00,0.00,197.80,SD-1010-02,231.00,408.00,597.00,6.00,210.96,0.00,53.28,66.06,99.81,386.18,227.67,547.00,1.03,164.19,230.25,0.00,2,1,6,0.87,0.50,0.66,0.75
4,2026-02-09,WH-1010-01,MAT001,ACEH,61.10,59.50,333.72,SD-1010-02,122.50,229.00,529.00,4.00,0.00,210.96,0.00,52.74,105.48,197.80,231.00,597.00,0.00,173.51,226.25,1.03,2,1,7,0.87,0.50,0.75,0.66


In [28]:
master_ts.to_csv('data_for_optimizing.csv', index=False)
print("Tersimpan: data_for_optimizing.csv")

Tersimpan: data_for_optimizing.csv



#### Ringkasan kolom final

##### Dataset Revenue Forecasting (`data_for_forecast_revenue.csv`)

| Grup | Kolom | Keterangan |
|------|-------|------------|
| Identifier | `soldto`, `periode`, `regional`, `province_desc` | Kunci entitas dan waktu |
| **Target** | `target_revenue` | Variabel dependen (Y) |
| Harga & kapasitas | `avg_price_per_ton`, `utilisasi_kapasitas` | Kondisi harga dan utilisasi |
| **Lag revenue** | `lag_1w_revenue`, `lag_4w_revenue`, `rolling_4w_revenue_mean/std` | Autoregressive features — terpenting |
| Sell-out lag | `lag_1w_sellout`, `lag_1w_toko_aktif`, `lag_1w_sell_through_rate`, `rolling_4w_str_mean` | Sinyal daya serap, dilag 1 minggu |
| CA | `total_kapasitas_tonase`, `jumlah_gudang_distributor` | Upper bound fisik |
| **Survey** | `lag_1w_avg_stock_toko`, `lag_1w_volume_sales/order`, `lag_1w_n_toko_surveyed` | Sinyal permintaan riil lapangan |
| Musiman | `month_sin/cos`, `week_sin/cos`, `quarter` | Pola waktu |

##### Dataset Optimalisasi Stok (`data_for_optimizing.csv`)

| Grup | Kolom | Keterangan |
|------|-------|------------|
| Identifier | `kode_gudang`, `material_code`, `week_start`, `provinsi` | Kunci panel |
| **Target** | `actual_tonase_in` | Volume riil masuk gudang per material |
| Alokasi | `target_tonase_ca` | Target kapasitas dari CA |
| Lag aktual | `lag_1w/2w/4w_actual`, `rolling_4w_actual_mean/std` | Autoregressive, shift sebelum rolling |
| Demand hilir | `lag_1w_sellout_gudang` | Total sell-out gudang tanpa breakdown material |
| Fulfillment | `lag_1w_fulfillment_ratio` | Utilisasi vs target, dilag |
| **Safety stock** | `safety_stock_est`, `reorder_point_est` | Komponen SS = Z × σ × √LT |
| **Survey** | `lag_1w_stock_toko`, `lag_1w_vol_order_dist` | Sinyal toko via distributor |
| Musiman | `month_sin/cos`, `week_sin/cos`, `quarter` | Pola waktu |